In [ ]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go


In [ ]:
nodes = []
nodal_values = np.loadtxt('../text_files/nodes_minimal_model.txt', delimiter=',')
theta = -15 # degrees

nodal_values[:, 1] = nodal_values[:, 1] * np.cos(np.radians(theta)) - nodal_values[:, 2] * np.sin(np.radians(theta))
nodal_values[:, 2] = nodal_values[:, 1] * np.sin(np.radians(theta)) + nodal_values[:, 2] * np.cos(np.radians(theta))

for i in range(nodal_values.shape[0]):
  nodes.append(n.nodes(nodal_values[i, 0], nodal_values[i, 1], nodal_values[i, 2]))



# **Primary truss parameters**

In [ ]:
# Truss parameters
E = 210e9  # Young's modulus in Pascals
nu = 0.3   # Poisson's ratio
G = E / (2 * (1 + nu))  # Shear modulus in Pascals
m = 6.7504 * 10 ** 6 #kg
m = 5596024
d1 = 0.8 #m
d2 = 1.8 #m
t1 = 0.03 #m
t2 = 0.08 #m
L_truss_element_y = 15 #m
h_truss = 18 #m
Aeq, Ieqy, Ieqz, b_eq, h_eq = mat.effective_truss_stiffness(d1, d2, t1, t2, h_truss, L_truss_element_y) 
L = 237.5
rho_truss = m / (L * Aeq)
It = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
k = 0.08
Ip  = Ieqy + Ieqz
ep_K = [E, G, Aeq, Ieqy, Ieqz, It, k]
ep_m = [rho_truss, Aeq, Ieqy, Ieqz, Ip]

# **Fenders, connecting elements and retaining wall parameters**

In [ ]:
k = 5/6
rho = 7850 #kg/m^3
E=210e9 #Pa
G = E/(2*(1+nu)) #Pa
nu = 0.3 #Poisson's ratio
k_fender =  mat.stiffness_fenders()
k_fender = 5e6
Iy_connect, Iz_connect, Ip_connect, It_connect, A_connect = mat.stiffness_connecting_beams()
ep_K_connect = [E, G, A_connect, Iy_connect, Iz_connect, It_connect, k]
ep_m_connect = [rho, A_connect, Iy_connect, Iz_connect, Ip_connect]



h_eq = 22
Iy_wall = 1250
Iz_wall = 0.25 * Iy_wall
Ip_wall = Iy_wall + Iz_wall
It_wall = 0.05 * Iy_wall
A_wall = Iy_wall * 12 / h_eq**2
m_wall = 6455148
L_wall = 237.5
rho_wall = m_wall / (L_wall * A_wall)
ep_K_wall = [E, G, A_wall, Iy_wall, Iz_wall, It_wall, k]
ep_m_wall = [rho_wall, A_wall, Iy_wall, Iz_wall, Ip_wall]


# **Connecting truss parameters**

In [ ]:
A_eq, Iy_eq, Iz_eq, b_eq, h_eq = mat.stiffness_connecting_truss(d1, d2, t1, t2, h_truss, L_truss_element_y)
Ip_connecting_truss = Iy_eq + Iz_eq 
It_eq = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
ep_K_connecting_truss = [E, G, A_eq, Iy_eq, Iz_eq, It_eq, k]
ep_m_connecting_truss = [rho_truss, A_eq, Iy_eq, Iz_eq, Ip_connect]

# **Braces parameters**

In [ ]:
# braces
k = 5/6
Iy, Iz, Ip, It, A = mat.stiffness_braces()
ep_K_braces = [E, G, A, Iy, Iz, It, k]
ep_m_braces = [rho, A, Iy, Iz, Ip]

# **Assembling the material properties to the correct elements**

In [ ]:
elements = []
element_nodes = np.loadtxt('../text_files/element_nodes.txt', dtype=int)
for i in range(element_nodes.shape[0]):
    if element_nodes[i, 2] == 0:
        elements.append(e.elements_wet(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K, ep_m, theta))
    elif element_nodes[i, 2] == 1:
        elements.append(e.elements_wet(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_wall, ep_m_wall, theta))
    elif element_nodes[i, 2] == 2:
        elements.append(e.elements_wet(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_connect, ep_m_connect, theta))
    elif element_nodes[i, 2] == 3:
        elements.append(e.elements_wet(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_connecting_truss, ep_m_connecting_truss, theta))   
    elif element_nodes[i, 2] == 4:
        elements.append(e.elements_wet(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_braces, ep_m_braces, theta))

element_nodes = element_nodes[:, :2] # Remove the column with element type information


dofs = n.degrees_of_freedom(nodes)

element_locs = []

for (nA, nB) in element_nodes:
    dofs_A = dofs[f'dof_{nA}']
    dofs_B = dofs[f'dof_{nB}']
    element_locs.append(np.hstack((dofs_A, dofs_B)))



In [ ]:
N = len(nodes)
DOFS_per_node = 6



K_global = np.zeros((N * DOFS_per_node, N * DOFS_per_node))
M_global = np.zeros((N * DOFS_per_node, N * DOFS_per_node))

K_locs = []
M_locs = []



for i in range(len(element_locs)):
    K_global[np.ix_(element_locs[i], element_locs[i])] += elements[i][-1]
    M_global[np.ix_(element_locs[i], element_locs[i])] += elements[i][-2]



# To restore symmetry of the global stiffness and mass matrices
K_global = 0.5 * (K_global + K_global.T)
M_global = 0.5 * (M_global + M_global.T)



In [ ]:
from matplotlib.lines import Line2D

def plot_elements2d(elements):

    fig, ax = plt.subplots(figsize=(8, 8))

    for i, element in enumerate(elements):
        n1, n2, _, _ = element
        x = [n1[0], n2[0]]
        y = [n1[1], n2[1]]

        if i < 60:
            color = 'red'
        elif i < 70:
            color = 'orange'
        elif i < 130:
            color = 'green'
        elif i < 169:
            color = 'blue'
        elif i > 169:
            color = 'grey'
        # Plot line
        ax.plot(x, y, color=color)

        # Plot nodes
        ax.scatter(*n1[:2], color=color, s=20)
        ax.scatter(*n2[:2], color=color, s=20)
    
    for i, element in enumerate(elements):
        n1, n2, _, _ = element
        x = [n1[0], n2[0]]
        y = [n1[1], n2[1]]

        if i == 130 or i == 140 or i == 150 or i == 160:
            color = 'red'
            ax.scatter(*n1[:2], color=color, s=20)

        elif i == 139 or i == 149 or i == 159 or i == 169:
            color = 'green'
            ax.scatter(*n2[:2], color=color, s=20)
        elif i == 170 or i == 185:
            color = 'red'
            ax.scatter(*n1[:2], color=color, s=20)




    # ---- Custom legend: line + ball marker ----
    legend_handles = [
        Line2D([0], [0], color='red', marker='o', markersize=6, label='Primary trusses'),
        Line2D([0], [0], color='orange', marker='o', markersize=6, label='Connecting truss'),
        Line2D([0], [0], color='green', marker='o', markersize=6, label='Retaining wall'),
        Line2D([0], [0], color='blue', marker='o', markersize=6, 
               label='Connection trusses and retaining wall'),
        Line2D([0], [0], color='grey', marker='o', markersize=6, label='Braces for connecting elements')

    ]

    ax.legend(handles=legend_handles, loc='lower right')

    ax.set_xlabel('Length [m]')
    ax.set_ylabel('Length [m]')
    ax.set_aspect('equal')
    ax.set_title('Nodes and elements of the minimal model of the Maeslantkering')
    plt.savefig('../../elements_2d.png', dpi=500)

    plt.show()

plot_elements2d(elements)

In [ ]:
N = len(nodes)
DOFS_per_node = 6



K_global = np.zeros((N * DOFS_per_node, N * DOFS_per_node))
M_global = np.zeros((N * DOFS_per_node, N * DOFS_per_node))
print(K_global.shape    )

K_locs = []
M_locs = []



for i in range(len(element_locs)):
    K_global[np.ix_(element_locs[i], element_locs[i])] += elements[i][-1]
    M_global[np.ix_(element_locs[i], element_locs[i])] += elements[i][-2]



# To restore symmetry of the global stiffness and mass matrices
K_global = 0.5 * (K_global + K_global.T)
M_global = 0.5 * (M_global + M_global.T)



In [ ]:
fender_dofs =  [70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 99,
               100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124,
               126, 128]
fender_dofs = [70, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 128]

k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
for dof in k_fender_dofs:
    K_global[dof, dof] += k_fender



In [ ]:
indices_to_remove = np.hstack((dofs['dof_1'][0:3], dofs['dof_129'][0:2]))
keep_indices = np.setdiff1d(np.arange(N * DOFS_per_node), indices_to_remove)
K_global_reduced = K_global[np.ix_(keep_indices, keep_indices)]
M_global_reduced = M_global[np.ix_(keep_indices, keep_indices)]

In [ ]:
eigvals_global, eigvecs_global = eigh(K_global_reduced, M_global_reduced)


tol = 1e-6
positive = eigvals_global > tol
eigvals_global = eigvals_global[positive]
eigvecs_global = eigvecs_global[:, positive]

frequencies_rad = np.sqrt(eigvals_global)
frequencies_hz = frequencies_rad / (2 * np.pi)


data_freqs = pd.DataFrame({
    'Frequency (Hz)': frequencies_hz,
    'Frequency (rad/s)': frequencies_rad
})

display(data_freqs[:15])


In [ ]:
eigvecs_full = e.expand_eigenvectors(eigvecs_global, keep_indices, N*DOFS_per_node)
print("Expanded eigenvectors shape:", eigvecs_global.shape)
np.save('../text_files/canal_eigvecs_global.npy', eigvecs_full)

In [ ]:
eigvecs_disp_full = e.extract_displacement(eigvecs_full, keep=3, skip=3)
eigvecs_full.shape


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ==============================================================
# 1. COLOR RULES (EXACTLY LIKE YOUR MATPLOTLIB 2D PLOT)
# ==============================================================

def element_color(i):
    if i < 60:
        return "red"
    elif i < 70:
        return "orange"
    elif i < 130:
        return "green"
    elif i < 169:
        return "blue"
    else:
        return "grey"


# ---- Assign element colors ----
element_colors = [element_color(i) for i in range(len(element_nodes))]


# ==============================================================
# 2. NODE COLORS (EXACT SAME LOGIC AS YOUR SPECIAL RULES)
# ==============================================================

node_colors = ["black"] * len(nodes)   # default

for i, (nA, nB) in enumerate(element_nodes):

    col = element_colors[i]

    # Base rule: nodes inherit element color
    node_colors[nA - 1] = col
    node_colors[nB - 1] = col

    # ---- SPECIAL COLORING RULES (your exact code) ----
    if i in [130, 140, 150, 160]:   # node1 becomes red
        node_colors[nA - 1] = "red"

    if i in [139, 149, 159, 169]:   # node2 becomes green
        node_colors[nB - 1] = "green"

    if i in [170, 185]:             # node1 red
        node_colors[nA - 1] = "red"



# ==============================================================
# 3. STATIC 3D MODE SHAPE PLOTTER
# ==============================================================

def plot_mode_shape_3d(mode_index, scale=10000):

    # mode displacement vector reshaped to N×3
    mode_disp = eigvecs_disp_full[:, mode_index].reshape(-1, 3)

    # amplified deformed shape
    U = nodes + scale * mode_disp

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # ---- plot elements with their own colors ----
    for i, (nA, nB) in enumerate(element_nodes):
        p1 = U[nA - 1]
        p2 = U[nB - 1]

        ax.plot(
            [p1[0], p2[0]],
            [p1[1], p2[1]],
            [p1[2], p2[2]],
            color=element_colors[i],
            linewidth=2
        )

    # ---- plot nodes with their own colors ----
    ax.scatter(
        U[:,0],
        U[:,1],
        U[:,2],
        c=node_colors,
        s=25,
        # depthshade=True
    )

    # ---- labels ----
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_zlabel("Z [m]")
    ax.set_title(f"Mode Shape {mode_index + 1} (Static 3D View)")

    # ---- equal axis scaling ----
    max_range = np.array([
        U[:,0].max() - U[:,0].min(),
        U[:,1].max() - U[:,1].min(),
        U[:,2].max() - U[:,2].min()
    ]).max() / 2

    mid_x = (U[:,0].max() + U[:,0].min()) / 2
    mid_y = (U[:,1].max() + U[:,1].min()) / 2
    mid_z = (U[:,2].max() + U[:,2].min()) / 2

    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

    plt.tight_layout()
    plt.show()



# ==============================================================
# 4. HOW TO USE
# ==============================================================

# Example: plot mode 1 (index 0)
# plot_mode_shape_3d(0)

# Example: plot mode 6 (index 5)
plot_mode_shape_3d(5)

# Example: plot mode 12
# plot_mode_shape_3d(11)

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ---------------- SAME COLOR RULES ----------------
def element_color(i):
    if i < 60:
        return "red"
    elif i < 70:
        return "orange"
    elif i < 130:
        return "green"
    elif i < 169:
        return "blue"
    else:
        return "grey"

# List element colors
element_colors = [element_color(i) for i in range(len(element_nodes))]

# ---------------- NODE COLORS ----------------
node_colors = ["black"] * len(nodes)

for i, (nA, nB) in enumerate(element_nodes):

    col = element_colors[i]

    node_colors[nA - 1] = col
    node_colors[nB - 1] = col

    if i in [130, 140, 150, 160]:
        node_colors[nA - 1] = "red"
    if i in [139, 149, 159, 169]:
        node_colors[nB - 1] = "green"
    if i in [170, 185]:
        node_colors[nA - 1] = "red"


# ---------------- SETTINGS ----------------
modes_to_plot = 5
num_frames = 180
t = np.linspace(0, 2*np.pi, num_frames)
omega = 3.0
scale = 100000

nodes = np.array(nodes)
eigvecs_disp_full = np.array(eigvecs_disp_full)

# -------- FIXED AXES --------
min_x, max_x = nodes[:,0].min(), nodes[:,0].max()
min_y, max_y = nodes[:,1].min(), nodes[:,1].max()
min_z, max_z = nodes[:,2].min(), nodes[:,2].max()

Lmax = max(max_x-min_x, max_y-min_y, max_z-min_z)

mid_x = (min_x + max_x)/2
mid_y = (min_y + max_y)/2
mid_z = (min_z + max_z)/2

xr = [mid_x - Lmax/2, mid_x + Lmax/2]
yr = [min_y - 10, max_y + 10]
zr = [mid_z - Lmax/2, mid_z + Lmax/2]


# ---------------- PLOT MODES ----------------
for i in range(modes_to_plot):

    mode_disp = eigvecs_disp_full[:, i].reshape(-1, 3)
    frames = []

    for k in range(num_frames):

        U = nodes + scale * np.sin(omega * t[k]) * mode_disp
        frame_traces = []

        # ---- elements ----
        for e, (nA, nB) in enumerate(element_nodes):
            p1 = U[nA - 1]
            p2 = U[nB - 1]

            frame_traces.append(
                go.Scatter3d(
                    x=[p1[0], p2[0]],
                    y=[p1[1], p2[1]],
                    z=[p1[2], p2[2]],
                    mode="lines",
                    line=dict(color=element_colors[e], width=5),
                    showlegend=False
                )
            )

        # ---- nodes ----
        frame_traces.append(
            go.Scatter3d(
                x=U[:,0],
                y=U[:,1],
                z=U[:,2],
                mode="markers",
                marker=dict(size=3, color=node_colors),
                showlegend=False
            )
        )

        frames.append(go.Frame(data=frame_traces))

    # INITIAL FRAME
    fig = go.Figure(data=frames[0].data, frames=frames)

    # -------- LEGEND TRACES (STATIC) --------
    legend_traces = [
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
                     line=dict(color="red", width=5),
                     name="Primary trusses"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
                     line=dict(color="orange", width=5),
                     name="Connecting truss"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
                     line=dict(color="green", width=5),
                     name="Retaining wall"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
                     line=dict(color="blue", width=5),
                     name="Connection trusses / retaining wall"),
        go.Scatter3d(x=[None], y=[None], z=[None], mode="lines",
                     line=dict(color="grey", width=5),
                     name="Braces for connecting elements")
    ]

    for tr in legend_traces:
        fig.add_trace(tr)

    # -------- LAYOUT --------
    fig.update_layout(
        title=f"Mode {i+1} – Animated Mode Shape",
        width=900,
        height=700,
        showlegend=True,
        legend=dict(x=0.85, y=0.15),
        scene=dict(
            xaxis=dict(title="X (m)", range=xr),
            yaxis=dict(title="Y (m)", range=yr),
            zaxis=dict(title="Z (m)", range=zr),
            aspectmode="manual",
            aspectratio=dict(x=1, y=1, z=1)
        ),
        updatemenus=[
            {
                "type": "buttons",
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args":[None, {"frame": {"duration": 40, "redraw": True},
                                   "transition": {"duration": 0}}]
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args":[[None], {"mode": "immediate"}]
                    }
                ]
            }
        ]
    )

    fig.show()